# Phutball: gated GRPO from v4 r14 (cost-sensitive reward)

Expert iteration (v3 / v4: `curriculum_v4_colab.ipynb`) only ever trains on correct samples, so it cannot penalise a
mistake; the training mix alone decided which error the model drifted into (v4 r7-r13 said NO WIN on real wins,
r15-r16 claimed wins that do not work). This phase is RL with an explicit cost for each mistake
(`grpo_pilot --reward cost`, `llm_train/grpo_reward.PhutballCostReward`):

| outcome | reward |
|---|---|
| correct (win or NO WIN) | +1 - lam * tokens (lam 3e-5, ramped in) |
| false win claim / wrong move | -a (1.0) |
| NO WIN on a real win | -b (1.5) |
| unparsable / self-chosen NOT FOUND | -c (0.25) |
| cut off -> forced ("... if you have not found one, answer NOT FOUND") | correct +0.5, NOT FOUND 0, wrong as above |

**Phases**: each trains `STEPS` GRPO steps (4 puzzles x 8 samples) from the current best adapter, on the puzzles the v4
curriculum found learnable (active: 0 < finished successes < 8), then the validation set (`val_v3`, never trained on)
and **gate v3.3** decide: promoted unless finished accuracy is clearly worse than the best or the peak, forced accuracy
collapses, or either error (false NO WIN on real wins, false win claims; finished or forced) clearly grows. A rejected
phase is discarded and the next one restarts from the best. Start: best = v4 r14 (0.459), peak = v4 r6 (0.474).
**Runtime:** A100, High-RAM. Resumable: re-run all cells after a disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request, gzip, shutil
V4 = '/content/drive/MyDrive/phutball/curriculum_v4'                      # the expert-iteration run (start + puzzles)
OUT = '/content/drive/MyDrive/phutball/grpo_cost'; os.makedirs(OUT, exist_ok=True)
INIT = f'{V4}/r14/adapter/final'                                            # v4's best (gate v3.2), validation 0.459
POOL = 'llm_train/puzzles/pool_v4.jsonl.gz'; FMT = 'ascii+men'
TEST = 'llm_bench/data/bench_v2.jsonl'; VAL = 'llm_train/puzzles/val_v3.jsonl'
STATE = f'{OUT}/gate_state.json'; PUZ = f'{OUT}/grpo_puzzles.jsonl'
CAP = 6144; PG = 25; EK = 4
STEPS = 20; PHASES = 16
TARGET = True                                 # llm_train.grpo_targets: weight buckets by forced - finished gap                       # ~2-3 h per phase (GRPO + the validation eval); raise PHASES any time
COST = '--cost-a 1.0 --cost-b 1.5 --cost-c 0.25 --lam 3e-5 --lam-ramp 50 --force-ok 0.0 --cut-cost 0.35'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model, samples=EK, bench=TEST, pg=PG):
    """bench: the test set (default) or VAL (gate: all 1,100 items, pg=0)."""
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --bench {bench} --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples {samples} --max-tokens {CAP} --per-group {pg} --concurrency 96 | tail -14
    !python -m llm_bench.run --bench {bench} --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {pg} --concurrency 48 | tail -14
    !cp llm_bench/results/{name}.jsonl llm_bench/results/{name}.summary.json llm_bench/results/{name}-forced.summary.json {EV}/
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or INIT

## 1. Start: best = v4 r14, peak = v4 r6; the GRPO puzzles

Copies v4's validation results of r14 (`v14` -> `v0`, the first best) and r6 (`v6` -> `peak`), and writes the puzzles
the v4 curriculum found learnable (status active) to `grpo_puzzles.jsonl` (the validation puzzles were never in the pool).

In [ ]:
for src, dst in (('v14', 'v0'), ('v6', 'peak')):
    for suf in ('.jsonl', '.summary.json', '-forced.summary.json'):
        if not os.path.exists(f'{EV}/{dst}{suf}'): shutil.copy(f'{V4}/eval/{src}{suf}', f'{EV}/{dst}{suf}')
if not os.path.exists(STATE):
    json.dump({"puzzles": {}, "rates": {}, "applied": [], "best": INIT, "best_eval": f'{EV}/v0', "peak_eval": f'{EV}/peak',
               "log": []}, open(STATE, 'w'))
if not os.path.exists(PUZ):
    st4 = json.load(open(f'{V4}/state.json')); active = {p for p, s in st4['puzzles'].items() if s['status'] == 'active'}
    with open(PUZ, 'w') as f:
        for l in gzip.open(POOL, 'rt'):
            if json.loads(l)['id'] in active: f.write(l)
from collections import Counter
print(sum(1 for _ in open(PUZ)), 'puzzles:', dict(Counter(json.loads(l)['subtask'] for l in open(PUZ))))
print('best:', best_adapter())

## 2. Gated GRPO phases (unattended)

In [ ]:
for P in range(1, PHASES + 1):
    d = f'{OUT}/p{P}'
    if not os.path.exists(f'{d}/final/adapter_model.safetensors'):
        print(f'=== phase {P}: GRPO from', best_adapter())
        puz = PUZ
        if TARGET:
            os.makedirs(d, exist_ok=True); puz = f'{d}/puzzles.jsonl'
            if not os.path.exists(puz):
                best_eval = json.load(open(STATE))['best_eval']
                !python -m llm_train.grpo_targets --puzzles {PUZ} --best-eval {best_eval} --rollouts "{OUT}/p*/rollouts.jsonl" --seed {P} --out {puz}
        !python -m llm_train.grpo_pilot --out {d} --init-adapter {best_adapter()} --puzzles {puz} --prompt-format {FMT} --reward cost {COST} --cap {CAP} --max-steps {STEPS} --hours 99 --save-every 5 --seed {P} 2>&1 | grep --line-buffered -E "puzzles|loaded adapter|reward call|cost reward|saved|rror|Traceback"
    if not os.path.exists(f'{EV}/v{P}-forced.summary.json'):
        server = serve({f'p{P}': f'{d}/final'})
        evaluate(f'v{P}', f'p{P}', samples=1, bench=VAL, pg=0)
        !pkill -f "vllm serve"
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum_pp gate --state {STATE} --round {P} --new {EV}/v{P} --adapter {d}/final | tee {d}/gate.txt
print('done; best:', best_adapter())

## 3. Results

In [ ]:
st = json.load(open(STATE))
for e in st['log']:
    if e['event'] == 'gate': print(f"phase {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
print('best:', st['best'])
# test set (benchmark) for the final best, once:
# server = serve({'best': best_adapter()}); evaluate('best-test', 'best'); !pkill -f "vllm serve"

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()